# 📊 Redlara Sheets Validation: Local DuckDB vs AWS Athena (Prod)

This notebook is **exclusively dedicated to validating the Redlara spreadsheets flow** across the Bronze and Silver layers between the local DuckDB database (`huntington_data_lake.duckdb`) and AWS Athena production (`bronze_redlara_prod`, `silver_redlara_prod`).

### Validation Scope:
1. **Bronze Layer Audit**: 12 local raw sheet tables (Ibirapuera, Santa Joana, Vila Mariana for 2021–2024) vs AWS Athena `bronze_redlara_prod` tables (`fet`, `fresh`, `fp`, `fto`, `iui`, `od`).
2. **Silver Layer Reconciliation**: Local `silver.redlara_unified` vs AWS Athena `silver_redlara_prod.fet`.
3. **Entity Resolution & Identity Audit**: Evaluation of `prontuario` resolution (`int_identity_resolution__redlara_fet`) and coverage.
4. **Clinical Metric & Outcome Parity**: Distribution of outcomes, procedures, deliveries, and quantitative clinical sums.
5. **1-to-1 Deterministic Row-Level Concordance**: Field-by-field value agreement across all 7,733 records.

### Rules:
- **Rule**: Each code cell performing queries explicitly opens and closes database connections.
- **Rule**: Never expose unmasked PII (patient names, CPF, dates of birth).


In [1]:
import os
import duckdb
import pandas as pd
import numpy as np
from pyathena import connect
import warnings
warnings.filterwarnings('ignore')

# Database Configurations
DUCKDB_PATH = '../../database/huntington_data_lake.duckdb'
ATHENA_REGION = 'sa-east-1'
ATHENA_WORKGROUP = 'datalake-admins'
ATHENA_SILVER_DB = 'silver_redlara_prod'
ATHENA_BRONZE_DB = 'bronze_redlara_prod'

print("Configuration set successfully.")
print(f"Local DuckDB Path: {os.path.abspath(DUCKDB_PATH)}")
print(f"AWS Athena Silver Schema: {ATHENA_SILVER_DB}")
print(f"AWS Athena Bronze Schema: {ATHENA_BRONZE_DB}")


Configuration set successfully.
Local DuckDB Path: G:\My Drive\projetos_individuais\Huntington\database\huntington_data_lake.duckdb
AWS Athena Silver Schema: silver_redlara_prod
AWS Athena Bronze Schema: bronze_redlara_prod


In [ ]:
# Database Configuration
DUCKDB_PATH = "database/huntington_data_lake.duckdb"
ATHENA_REGION = "sa-east-1"
ATHENA_WORKGROUP = "datalake-admins"
ATHENA_DB = "silver_redlara_prod"


In [2]:
try:
    with duckdb.connect(DUCKDB_PATH, read_only=True) as conn:
        duck_ok = conn.execute("SELECT 1 as test").fetchone()[0] == 1
    print("✅ Local DuckDB Connection: OK")
except Exception as e:
    print(f"❌ Local DuckDB Connection: FAILED - {e}")
    duck_ok = False

try:
    with connect(region_name=ATHENA_REGION, work_group=ATHENA_WORKGROUP, schema_name=ATHENA_SILVER_DB) as conn:
        with conn.cursor() as cur:
            cur.execute("SELECT 1")
            athena_ok = cur.fetchone()[0] == 1
    print("✅ AWS Athena Connection: OK")
except Exception as e:
    print(f"❌ AWS Athena Connection: FAILED - {e}")
    athena_ok = False


✅ Local DuckDB Connection: OK
✅ AWS Athena Connection: OK


## 🟢 Part 1: Bronze Layer Inventory & Raw Ingestion Audit

Redlara raw data originates from annual Excel spreadsheets per unit (Ibirapuera, Santa Joana, Vila Mariana) covering years 2021–2024.
Here we audit the 12 local DuckDB bronze tables and the corresponding AWS Athena `bronze_redlara_prod` tables.


In [3]:
# 1. Bronze inventory in DuckDB
with duckdb.connect(DUCKDB_PATH, read_only=True) as conn:
    duck_bronze_tables = conn.execute("""
        SELECT table_name 
        FROM information_schema.tables 
        WHERE table_schema='bronze' AND table_name ILIKE '%redlara%'
        ORDER BY table_name
    """).fetchall()
    
    bronze_rows = []
    for t in duck_bronze_tables:
        t_name = t[0]
        cnt = conn.execute(f"SELECT count(*) FROM bronze.{t_name}").fetchone()[0]
        cols_cnt = len(conn.execute(f"SELECT * FROM bronze.{t_name} LIMIT 0").description)
        bronze_rows.append({'Local Bronze Table': f"bronze.{t_name}", 'Row Count': cnt, 'Columns': cols_cnt})

df_duck_bronze = pd.DataFrame(bronze_rows)
print(f"Total Local Bronze Redlara Tables: {len(df_duck_bronze)} (Total Rows: {df_duck_bronze['Row Count'].sum():,})")
display(df_duck_bronze)

# 2. Bronze inventory in Athena
with connect(region_name=ATHENA_REGION, work_group=ATHENA_WORKGROUP, schema_name=ATHENA_BRONZE_DB) as conn:
    with conn.cursor() as cur:
        cur.execute(f"SHOW TABLES IN {ATHENA_BRONZE_DB}")
        ath_bronze_tables = [r[0] for r in cur.fetchall() if not r[0].startswith('_dlt')]
        
    ath_bronze_rows = []
    for t in ath_bronze_tables:
        cnt = pd.read_sql(f"SELECT count(*) as c FROM {ATHENA_BRONZE_DB}.{t}", conn)['c'][0]
        ath_bronze_rows.append({'Athena Bronze Table': f"{ATHENA_BRONZE_DB}.{t}", 'Row Count': cnt})

df_ath_bronze = pd.DataFrame(ath_bronze_rows)
print(f"\nAthena Bronze Redlara Datasets:")
display(df_ath_bronze)


Total Local Bronze Redlara Tables: 12 (Total Rows: 7,733)

Athena Bronze Redlara Datasets:


,Local Bronze Table,Row Count,Columns
0,bronze.redlara_ibirapuera_2021,971,63
1,bronze.redlara_ibirapuera_2022,776,64
2,bronze.redlara_ibirapuera_2023,884,68
3,bronze.redlara_ibirapuera_2024,847,70
4,bronze.redlara_sj_2021,465,66
5,bronze.redlara_sj_2022,589,71
6,bronze.redlara_sj_2023,400,72
7,bronze.redlara_sj_2024,410,92
8,bronze.redlara_vm_2021,641,62
9,bronze.redlara_vm_2022,645,99


,Athena Bronze Table,Row Count
0,bronze_redlara_prod.fet,270655
1,bronze_redlara_prod.fp,115640
2,bronze_redlara_prod.fresh,274365
3,bronze_redlara_prod.fto,70840
4,bronze_redlara_prod.iui,4830
5,bronze_redlara_prod.od,26670


## 🔵 Part 2: Silver Layer Reconciliation (`silver.redlara_unified` vs `silver_redlara_prod.fet`)

### 2.1 Schema & Field Inventory
Comparing column inventories, metadata columns, and privacy/PII handling.


In [4]:
with duckdb.connect(DUCKDB_PATH, read_only=True) as conn:
    local_cols = [col[0].lower() for col in conn.execute("SELECT * FROM silver.redlara_unified LIMIT 0").description]

with connect(region_name=ATHENA_REGION, work_group=ATHENA_WORKGROUP, schema_name=ATHENA_SILVER_DB) as conn:
    prod_cols = [col.lower() for col in pd.read_sql("SELECT * FROM fet LIMIT 0", conn).columns]

only_local = sorted(list(set(local_cols) - set(prod_cols)))
only_prod = sorted(list(set(prod_cols) - set(local_cols)))
common_cols = sorted(list(set(local_cols) & set(prod_cols)))

df_schema_summary = pd.DataFrame([
    {'Category': 'Total Columns (Local DuckDB)', 'Count': len(local_cols), 'Details': ', '.join(local_cols)},
    {'Category': 'Total Columns (Athena Prod)', 'Count': len(prod_cols), 'Details': ', '.join(prod_cols)},
    {'Category': 'Common Overlapping Columns', 'Count': len(common_cols), 'Details': ', '.join(common_cols)},
    {'Category': 'Local Only (PII Retained)', 'Count': len(only_local), 'Details': ', '.join(only_local)},
    {'Category': 'Athena Only (Pipeline Metadata)', 'Count': len(only_prod), 'Details': ', '.join(only_prod)}
])
display(df_schema_summary[['Category', 'Count']])
print("\nLocal Only Columns (PII):", only_local)
print("Athena Only Columns (DLT Ingestion Metadata):", only_prod)



Local Only Columns (PII): ['date_of_birth', 'patient_name']
Athena Only Columns (DLT Ingestion Metadata): ['_dlt_id', 'bronze_ingested_at', 'source_file']


,Category,Count
0,Total Columns (Local DuckDB),22
1,Total Columns (Athena Prod),23
2,Common Overlapping Columns,20
3,Local Only (PII Retained),2
4,Athena Only (Pipeline Metadata),3


### 2.2 Volume & Prontuário Identity Resolution
Validating total rows, distinct Redlara chart/PIN keys, and resolved patient prontuários.


In [5]:
with duckdb.connect(DUCKDB_PATH, read_only=True) as conn:
    local_vol = conn.execute("""
        SELECT 
            COUNT(*) as total_rows,
            COUNT(DISTINCT chart_or_pin) as unique_chart_pin,
            COUNT(DISTINCT prontuario) as unique_prontuario,
            COUNT(prontuario) as resolved_prontuarios,
            SUM(CASE WHEN prontuario IS NULL THEN 1 ELSE 0 END) as null_prontuarios
        FROM silver.redlara_unified
    """).df()

with connect(region_name=ATHENA_REGION, work_group=ATHENA_WORKGROUP, schema_name=ATHENA_SILVER_DB) as conn:
    prod_vol = pd.read_sql("""
        SELECT 
            COUNT(*) as total_rows,
            COUNT(DISTINCT chart_or_pin) as unique_chart_pin,
            COUNT(DISTINCT prontuario) as unique_prontuario,
            COUNT(prontuario) as resolved_prontuarios,
            SUM(CASE WHEN prontuario IS NULL THEN 1 ELSE 0 END) as null_prontuarios
        FROM fet
    """, conn)

df_vol = pd.DataFrame({
    'Metric': [
        'Total Rows',
        'Unique chart_or_pin',
        'Unique Resolved Prontuario',
        'Resolved Prontuarios (Non-Null)',
        'Null Prontuarios'
    ],
    'Local (DuckDB)': [
        local_vol.loc[0, 'total_rows'],
        local_vol.loc[0, 'unique_chart_pin'],
        local_vol.loc[0, 'unique_prontuario'],
        local_vol.loc[0, 'resolved_prontuarios'],
        local_vol.loc[0, 'null_prontuarios']
    ],
    'Athena (Prod)': [
        prod_vol.loc[0, 'total_rows'],
        prod_vol.loc[0, 'unique_chart_pin'],
        prod_vol.loc[0, 'unique_prontuario'],
        prod_vol.loc[0, 'resolved_prontuarios'],
        prod_vol.loc[0, 'null_prontuarios']
    ]
})
df_vol['Delta'] = df_vol['Local (DuckDB)'] - df_vol['Athena (Prod)']
df_vol['Match Rate %'] = (1.0 - (df_vol['Delta'].abs() / df_vol['Local (DuckDB)'])) * 100.0
display(df_vol)


,Metric,Local (DuckDB),Athena (Prod),Delta,Match Rate %
0,Total Rows,7733.0,7733,0.0,100.000000
1,Unique chart_or_pin,6050.0,6050,0.0,100.000000
2,Unique Resolved Prontuario,5229.0,5229,0.0,100.000000
3,Resolved Prontuarios (Non-Null),7696.0,7733,-37.0,99.519231
4,Null Prontuarios,37.0,0,37.0,0.000000


### 2.3 Dimensional Slicing: Year × Clinic (Unidade) Matrix
Auditing row volumes across all combinations of Year (2021–2024) and Clinic (Ibirapuera, Santa Joana, Vila Mariana).


In [6]:
with duckdb.connect(DUCKDB_PATH, read_only=True) as conn:
    df_loc = conn.execute("""
        SELECT 
            year,
            CASE 
                WHEN LOWER(unidade) LIKE '%ibirapuera%' THEN 'Ibirapuera'
                WHEN LOWER(unidade) IN ('vm', 'vila_mariana', 'vila mariana') THEN 'Vila Mariana'
                WHEN LOWER(unidade) IN ('sj', 'santa_joana', 'santa joana') THEN 'Santa Joana'
                ELSE unidade
            END as unidade_std,
            COUNT(*) as row_count
        FROM silver.redlara_unified
        GROUP BY 1, 2
    """).df()

with connect(region_name=ATHENA_REGION, work_group=ATHENA_WORKGROUP, schema_name=ATHENA_SILVER_DB) as conn:
    df_ath = pd.read_sql("""
        SELECT 
            year,
            CASE 
                WHEN LOWER(unidade) LIKE '%ibirapuera%' THEN 'Ibirapuera'
                WHEN LOWER(unidade) IN ('vm', 'vila_mariana', 'vila mariana') THEN 'Vila Mariana'
                WHEN LOWER(unidade) IN ('sj', 'santa_joana', 'santa joana') THEN 'Santa Joana'
                ELSE unidade
            END as unidade_std,
            COUNT(*) as row_count
        FROM fet
        GROUP BY 1, 2
    """, conn)

df_merged_slices = pd.merge(
    df_loc, df_ath, on=['year', 'unidade_std'], 
    how='outer', suffixes=(' (DuckDB)', ' (Athena)')
).fillna(0)

df_merged_slices['Delta'] = df_merged_slices['row_count (DuckDB)'] - df_merged_slices['row_count (Athena)']
df_merged_slices['Match Rate %'] = np.where(
    df_merged_slices['row_count (DuckDB)'] == df_merged_slices['row_count (Athena)'],
    100.0,
    (1.0 - (df_merged_slices['Delta'].abs() / df_merged_slices['row_count (DuckDB)'])) * 100.0
)
df_merged_slices = df_merged_slices.sort_values(by=['year', 'unidade_std']).reset_index(drop=True)
display(df_merged_slices)


,year,unidade_std,row_count (DuckDB),row_count (Athena),Delta,Match Rate %
0,2021,Ibirapuera,971,971,0,100.0
1,2021,Santa Joana,465,465,0,100.0
2,2021,Vila Mariana,641,641,0,100.0
3,2022,Ibirapuera,776,776,0,100.0
4,2022,Santa Joana,589,589,0,100.0
5,2022,Vila Mariana,645,645,0,100.0
6,2023,Ibirapuera,884,884,0,100.0
7,2023,Santa Joana,400,400,0,100.0
8,2023,Vila Mariana,601,601,0,100.0
9,2024,Ibirapuera,847,847,0,100.0


### 2.4 Clinical Outcome Distributions
Comparing categorical distributions for Redlara clinical outcomes between environments.


In [7]:
with duckdb.connect(DUCKDB_PATH, read_only=True) as conn:
    loc_outcomes = conn.execute("""
        SELECT 
            COALESCE(TRIM(outcome), 'NULL') as outcome_clean,
            COUNT(*) as loc_count
        FROM silver.redlara_unified
        GROUP BY 1
    """).df()

with connect(region_name=ATHENA_REGION, work_group=ATHENA_WORKGROUP, schema_name=ATHENA_SILVER_DB) as conn:
    ath_outcomes = pd.read_sql("""
        SELECT 
            COALESCE(TRIM(outcome), 'NULL') as outcome_clean,
            COUNT(*) as ath_count
        FROM fet
        GROUP BY 1
    """, conn)

df_outcomes = pd.merge(loc_outcomes, ath_outcomes, on='outcome_clean', how='outer').fillna(0)
df_outcomes['Delta'] = df_outcomes['loc_count'] - df_outcomes['ath_count']
df_outcomes['Match Rate %'] = np.where(
    df_outcomes['loc_count'] == df_outcomes['ath_count'],
    100.0,
    (1.0 - (df_outcomes['Delta'].abs() / df_outcomes['loc_count'])) * 100.0
)
df_outcomes = df_outcomes.sort_values(by='loc_count', ascending=False).reset_index(drop=True)

print("--- Top Clinical Outcome Categories ---")
display(df_outcomes)


--- Top Clinical Outcome Categories ---


,outcome_clean,loc_count,ath_count,Delta,Match Rate %
0,Embryo transfer,5505,5505,0,100.0
1,EMBRYO TRANSFER,1226,1226,0,100.0
2,Embryo Transfer,408,408,0,100.0
3,Cancellation,258,258,0,100.0
4,No embryo transfer,173,173,0,100.0
5,CANCELLATION,88,88,0,100.0
6,No Embryo Transfer,38,38,0,100.0
7,NULL,37,37,0,100.0


### 2.5 Quantitative Metric Aggregations & Proofs
Comparing statistical sums across all numerical clinical fields in Redlara.


In [8]:
with duckdb.connect(DUCKDB_PATH, read_only=True) as conn:
    local_metrics = conn.execute("""
        SELECT 
            SUM(TRY_CAST(number_of_newborns AS DOUBLE)) as sum_newborns,
            SUM(TRY_CAST(number_of_embryos_transferred AS DOUBLE)) as sum_transferred,
            SUM(TRY_CAST(n_of_normal AS DOUBLE)) as sum_normal,
            SUM(TRY_CAST(n_of_biopsied AS DOUBLE)) as sum_biopsied,
            SUM(TRY_CAST(gestational_age_at_delivery AS DOUBLE)) as sum_gest_age,
            SUM(TRY_CAST(baby_1_weight AS DOUBLE)) as sum_baby1_weight,
            SUM(TRY_CAST(number_of_fet_after_originally_frozen AS DOUBLE)) as sum_fet_after_frozen
        FROM silver.redlara_unified
    """).df()

with connect(region_name=ATHENA_REGION, work_group=ATHENA_WORKGROUP, schema_name=ATHENA_SILVER_DB) as conn:
    prod_metrics = pd.read_sql("""
        SELECT 
            SUM(TRY_CAST(number_of_newborns AS DOUBLE)) as sum_newborns,
            SUM(TRY_CAST(number_of_embryos_transferred AS DOUBLE)) as sum_transferred,
            SUM(TRY_CAST(n_of_normal AS DOUBLE)) as sum_normal,
            SUM(TRY_CAST(n_of_biopsied AS DOUBLE)) as sum_biopsied,
            SUM(TRY_CAST(gestational_age_at_delivery AS DOUBLE)) as sum_gest_age,
            SUM(TRY_CAST(baby_1_weight AS DOUBLE)) as sum_baby1_weight,
            SUM(TRY_CAST(number_of_fet_after_originally_frozen AS DOUBLE)) as sum_fet_after_frozen
        FROM fet
    """, conn)

metrics_list = [
    ('Sum number_of_newborns', 'sum_newborns'),
    ('Sum number_of_embryos_transferred', 'sum_transferred'),
    ('Sum n_of_normal', 'sum_normal'),
    ('Sum n_of_biopsied', 'sum_biopsied'),
    ('Sum gestational_age_at_delivery', 'sum_gest_age'),
    ('Sum baby_1_weight', 'sum_baby1_weight'),
    ('Sum number_of_fet_after_originally_frozen', 'sum_fet_after_frozen')
]

df_metrics_comp = pd.DataFrame({
    'Metric': [m[0] for m in metrics_list],
    'Local (DuckDB)': [local_metrics.loc[0, m[1]] for m in metrics_list],
    'Athena (Prod)': [prod_metrics.loc[0, m[1]] for m in metrics_list]
})
df_metrics_comp['Delta'] = df_metrics_comp['Local (DuckDB)'] - df_metrics_comp['Athena (Prod)']
df_metrics_comp['Match Rate %'] = np.where(
    df_metrics_comp['Delta'] == 0,
    100.0,
    (1.0 - (df_metrics_comp['Delta'].abs() / df_metrics_comp['Local (DuckDB)'])) * 100.0
)
display(df_metrics_comp)


,Metric,Local (DuckDB),Athena (Prod),Delta,Match Rate %
0,Sum number_of_newborns,2952.000,2952.000,0.0,100.000000
1,Sum number_of_embryos_transferred,9010.000,9010.000,0.0,100.000000
2,Sum n_of_normal,1135.000,1135.000,0.0,100.000000
3,Sum n_of_biopsied,2238.000,1772.000,466.0,79.177837
4,Sum gestational_age_at_delivery,93608.000,93608.000,0.0,100.000000
5,Sum baby_1_weight,7094627.623,7094627.623,0.0,100.000000
6,Sum number_of_fet_after_originally_frozen,2337.000,2337.000,0.0,100.000000


### 2.6 Deterministic 1-to-1 Row-Level Concordance (All 7,733 Records)

Because a single patient chart (`chart_or_pin`) may have multiple FET procedures within the same year/clinic, we sort both datasets deterministically by `(year, unidade, chart_or_pin, date_of_embryo_transfer, outcome, date_when_embryos_were_cryopreserved)` and perform an exact 1-to-1 row-by-row field comparison across all 7,733 records.


In [9]:
with duckdb.connect(DUCKDB_PATH, read_only=True) as conn:
    df_local_all = conn.execute("SELECT * FROM silver.redlara_unified").df()

with connect(region_name=ATHENA_REGION, work_group=ATHENA_WORKGROUP, schema_name=ATHENA_SILVER_DB) as conn:
    df_ath_all = pd.read_sql("SELECT * FROM fet", conn)

# Normalize and extract canonical string formats
for df in [df_local_all, df_ath_all]:
    df['chart_clean'] = df['chart_or_pin'].astype(str).str.strip().str.replace(r'\.0$', '', regex=True).replace({'nan': '', 'None': '', '<NA>': ''})
    df['unidade_clean'] = df['unidade'].astype(str).str.lower().replace({
        'vm': 'vila_mariana', 'vila mariana': 'vila_mariana',
        'sj': 'santa_joana', 'santa joana': 'santa_joana',
        'ibirapuera': 'ibirapuera'
    })
    df['year_clean'] = pd.to_numeric(df['year'], errors='coerce').fillna(-1).astype(int)
    df['outcome_clean'] = df['outcome'].astype(str).str.strip().str.lower().replace({'nan': '', 'none': '', '<na>': ''})
    df['date_et_clean'] = pd.to_datetime(df['date_of_embryo_transfer'], errors='coerce').dt.strftime('%Y-%m-%d').fillna('')
    df['date_del_clean'] = pd.to_datetime(df['date_of_delivery'], errors='coerce').dt.strftime('%Y-%m-%d').fillna('')
    df['date_cryo_clean'] = pd.to_datetime(df['date_when_embryos_were_cryopreserved'], errors='coerce').dt.strftime('%Y-%m-%d').fillna('')

sort_cols = ['year_clean', 'unidade_clean', 'chart_clean', 'date_et_clean', 'outcome_clean', 'date_cryo_clean']
df_loc_sorted = df_local_all.sort_values(by=sort_cols).reset_index(drop=True)
df_ath_sorted = df_ath_all.sort_values(by=sort_cols).reset_index(drop=True)

compare_fields = [
    ('chart_or_pin', 'chart_clean', 'chart_clean'),
    ('year', 'year_clean', 'year_clean'),
    ('unidade', 'unidade_clean', 'unidade_clean'),
    ('outcome', 'outcome_clean', 'outcome_clean'),
    ('outcome_type', 'outcome_type', 'outcome_type'),
    ('date_of_embryo_transfer', 'date_et_clean', 'date_et_clean'),
    ('date_of_delivery', 'date_del_clean', 'date_del_clean'),
    ('date_when_embryos_were_cryopreserved', 'date_cryo_clean', 'date_cryo_clean'),
    ('number_of_newborns', 'number_of_newborns', 'number_of_newborns'),
    ('number_of_embryos_transferred', 'number_of_embryos_transferred', 'number_of_embryos_transferred'),
    ('n_of_normal', 'n_of_normal', 'n_of_normal'),
    ('type_of_delivery', 'type_of_delivery', 'type_of_delivery'),
    ('gestational_age_at_delivery', 'gestational_age_at_delivery', 'gestational_age_at_delivery'),
    ('prontuario', 'prontuario', 'prontuario')
]

concordance_rows = []
total_rows = len(df_loc_sorted)

for label, col_loc, col_ath in compare_fields:
    s_loc = df_loc_sorted[col_loc].astype(str).str.strip().str.lower().replace({'nan': '', 'none': '', '<na>': ''})
    s_ath = df_ath_sorted[col_ath].astype(str).str.strip().str.lower().replace({'nan': '', 'none': '', '<na>': ''})
    s_loc = s_loc.str.replace(r'\.0$', '', regex=True)
    s_ath = s_ath.str.replace(r'\.0$', '', regex=True)
    
    matches = (s_loc == s_ath).sum()
    rate = (matches / total_rows) * 100.0
    concordance_rows.append({
        'Field': label,
        'Matching Records': matches,
        'Total Records': total_rows,
        'Concordance %': rate
    })

df_concordance = pd.DataFrame(concordance_rows)
print(f"Deterministic 1-to-1 Row-Level Comparison (Total Rows: {total_rows:,}):")
display(df_concordance)


Deterministic 1-to-1 Row-Level Comparison (Total Rows: 7,733):


,Field,Matching Records,Total Records,Concordance %
0,chart_or_pin,7733,7733,100.000000
1,year,7733,7733,100.000000
2,unidade,7733,7733,100.000000
3,outcome,7733,7733,100.000000
4,outcome_type,7733,7733,100.000000
5,date_of_embryo_transfer,7733,7733,100.000000
6,date_of_delivery,7732,7733,99.987068
7,date_when_embryos_were_cryopreserved,7728,7733,99.935342
8,number_of_newborns,6982,7733,90.288374
9,number_of_embryos_transferred,7727,7733,99.922410
